# 7. Neo4j Graph Ingestion

This notebook reads the enriched `clustered_emr.csv` (with `action_canonical` and `symptom_canonical` columns from notebooks 5 & 6) and builds the knowledge graph in Neo4j.

**Prerequisites:**
- Neo4j must be running (via `podman-compose up -d` in the `docker/` folder)
- Notebooks 1, 5, and 6 must have been run

**Graph Schema:**
- 7 Node types: EMRRecord, ProblemCluster, SymptomPattern, ActionPattern, Component, MachineModel, Part
- 9+ Relationship types with frequency-weighted aggregate edges

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from neo4j import GraphDatabase

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


In [2]:
print("1. Loading Enriched Data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

# Verify required columns exist
required = ["action_canonical", "symptom_canonical", "cluster_id", "cluster_label", "cause_canonical"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Run notebooks 1, 5, 6 first.")

print("Required columns verified.")
print(f"  Unique action patterns: {df['action_canonical'].nunique()}")
print(f"  Unique symptom patterns: {df['symptom_canonical'].nunique()}")
print(f"  Unique problem clusters: {df['cluster_id'].nunique()}")

1. Loading Enriched Data...
Loaded 20630 records.
Required columns verified.
  Unique action patterns: 942
  Unique symptom patterns: 1097
  Unique problem clusters: 1086


In [3]:
print("2. Connecting to Neo4j...")
driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password)
)

# Test connection
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(f"  Connection OK: {result.single()['test']}")

2. Connecting to Neo4j...
  Connection OK: 1


In [4]:
print("3. Clearing existing graph and creating constraints...")

with driver.session() as session:
    # Clear all existing data
    session.run("MATCH (n) DETACH DELETE n")
    print("  Cleared existing graph data.")
    
    # Create uniqueness constraints
    constraints = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:ProblemCluster) REQUIRE n.cluster_id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:SymptomPattern) REQUIRE n.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:ActionPattern) REQUIRE n.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:RootCausePattern) REQUIRE n.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Component) REQUIRE n.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:MachineModel) REQUIRE n.model IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Part) REQUIRE n.part_no IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:EMRRecord) REQUIRE n.emr_name IS UNIQUE",
    ]
    for c in constraints:
        session.run(c)
    print(f"  Created {len(constraints)} uniqueness constraints.")

3. Clearing existing graph and creating constraints...
  Cleared existing graph data.
  Created 8 uniqueness constraints.


In [5]:
print("4. Ingesting nodes and relationships...")

def safe_str(val, default=""):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    return str(val).strip()

def extract_model_family(model):
    if not model:
        return ""
    return str(model).split("-")[0]

batch_size = 500
total = len(df)
ingested = 0

for start in range(0, total, batch_size):
    batch = df.iloc[start:start+batch_size]
    
    with driver.session() as session:
        for _, row in batch.iterrows():
            emr_name = safe_str(row.get("EMR Name"))
            if not emr_name:
                continue
            
            # Core identifiers
            cluster_id = int(row.get("cluster_id", -1))
            cluster_label = safe_str(row.get("cluster_label"), "Uncategorized")
            symptom_canonical = safe_str(row.get("symptom_canonical"), "Uncategorized")
            action_canonical = safe_str(row.get("action_canonical"), "Uncategorized")
            cause_canonical = safe_str(row.get("cause_canonical"), "Penyebab Tidak Terdefinisi")
            machine_model = safe_str(row.get("Machine Model"))
            machine_product = safe_str(row.get("Machine Product"))
            serial_number = safe_str(row.get("Serial Number"))
            site = safe_str(row.get("Branch / Site"))
            raw_symptom = safe_str(row.get("Symptom"))
            raw_cause = safe_str(row.get("Caused of Problem"))
            created_date = safe_str(row.get("Created Date"))
            closed_date = safe_str(row.get("EMR Last Closed Date"))
            
            # Find action column
            raw_action = ""
            for col in row.index:
                if 'action' in col.lower() and 'corrected' in col.lower():
                    raw_action = safe_str(row.get(col))
                    break
            
            # Component and Part
            tc_component = safe_str(row.get("Techcare Component"))
            part_no = safe_str(row.get("Main Cause Part No"))
            part_desc = safe_str(row.get("Part Description"))
            
            # Build and run the Cypher query
            cypher = """
            MERGE (emr:EMRRecord {emr_name: $emr_name})
            SET emr.serial_number = $serial_number,
                emr.site = $site,
                emr.raw_symptom = $raw_symptom,
                emr.raw_cause = $raw_cause,
                emr.raw_action = $raw_action,
                emr.created_date = $created_date,
                emr.closed_date = $closed_date
            """
            params = {
                "emr_name": emr_name, "serial_number": serial_number,
                "site": site, "raw_symptom": raw_symptom,
                "raw_cause": raw_cause, "raw_action": raw_action,
                "created_date": created_date, "closed_date": closed_date,
            }
            
            # ProblemCluster
            if cluster_id != -1:
                cypher += """
                MERGE (pc:ProblemCluster {cluster_id: $cluster_id})
                SET pc.label = $cluster_label
                MERGE (emr)-[:BELONGS_TO]->(pc)
                """
                params["cluster_id"] = cluster_id
                params["cluster_label"] = cluster_label
            
            # SymptomPattern
            if symptom_canonical and symptom_canonical != "Uncategorized":
                cypher += """
                MERGE (sp:SymptomPattern {name: $symptom_canonical})
                MERGE (emr)-[:EXHIBITED {raw_text: $raw_symptom}]->(sp)
                """
                params["symptom_canonical"] = symptom_canonical
            
            # RootCausePattern
            if cause_canonical and cause_canonical != "Penyebab Tidak Terdefinisi":
                cypher += """
                MERGE (rc:RootCausePattern {name: $cause_canonical})
                MERGE (emr)-[:CAUSED_BY]->(rc)
                """
                params["cause_canonical"] = cause_canonical
            
            # ActionPattern
            if action_canonical and action_canonical != "Uncategorized":
                cypher += """
                MERGE (ap:ActionPattern {name: $action_canonical})
                MERGE (emr)-[:RESOLVED_BY {raw_text: $raw_action}]->(ap)
                """
                params["action_canonical"] = action_canonical
            
            # MachineModel
            if machine_model:
                cypher += """
                MERGE (mm:MachineModel {model: $machine_model})
                SET mm.family = $model_family, mm.product = $machine_product
                MERGE (emr)-[:ON_MACHINE]->(mm)
                """
                params["machine_model"] = machine_model
                params["model_family"] = extract_model_family(machine_model)
                params["machine_product"] = machine_product
            
            # Component
            if tc_component:
                cypher += """
                MERGE (c:Component {name: $tc_component})
                MERGE (emr)-[:AFFECTED]->(c)
                """
                params["tc_component"] = tc_component
            
            # Part
            if part_no:
                cypher += """
                MERGE (p:Part {part_no: $part_no})
                SET p.description = $part_desc
                MERGE (emr)-[:USED_PART]->(p)
                """
                params["part_no"] = part_no
                params["part_desc"] = part_desc
            
            session.run(cypher, **params)
    
    ingested += len(batch)
    print(f"  Ingested {ingested}/{total} records...")

print(f"\nNode ingestion complete: {ingested} records processed.")

4. Ingesting nodes and relationships...
  Ingested 500/20630 records...
  Ingested 1000/20630 records...
  Ingested 1500/20630 records...
  Ingested 2000/20630 records...
  Ingested 2500/20630 records...
  Ingested 3000/20630 records...
  Ingested 3500/20630 records...
  Ingested 4000/20630 records...
  Ingested 4500/20630 records...
  Ingested 5000/20630 records...
  Ingested 5500/20630 records...
  Ingested 6000/20630 records...
  Ingested 6500/20630 records...
  Ingested 7000/20630 records...
  Ingested 7500/20630 records...
  Ingested 8000/20630 records...
  Ingested 8500/20630 records...
  Ingested 9000/20630 records...
  Ingested 9500/20630 records...
  Ingested 10000/20630 records...
  Ingested 10500/20630 records...
  Ingested 11000/20630 records...
  Ingested 11500/20630 records...
  Ingested 12000/20630 records...
  Ingested 12500/20630 records...
  Ingested 13000/20630 records...
  Ingested 13500/20630 records...
  Ingested 14000/20630 records...
  Ingested 14500/20630 recor

In [6]:
print("5. Computing Aggregate Edges...")

with driver.session() as session:
    # SymptomPattern -> ProblemCluster (INDICATES)
    print("  Computing INDICATES edges (SymptomPattern -> ProblemCluster)...")
    session.run("""
        MATCH (sp:SymptomPattern)<-[:EXHIBITED]-(emr:EMRRecord)-[:BELONGS_TO]->(pc:ProblemCluster)
        WITH sp, pc, COUNT(emr) AS freq
        MERGE (sp)-[r:INDICATES]->(pc)
        SET r.frequency = freq
    """)
    
    # Compute strength (proportion of this symptom's records in this cluster)
    session.run("""
        MATCH (sp:SymptomPattern)<-[:EXHIBITED]-(emr:EMRRecord)
        WITH sp, COUNT(emr) AS total_for_symptom
        MATCH (sp)-[r:INDICATES]->(pc:ProblemCluster)
        SET r.strength = toFloat(r.frequency) / total_for_symptom
    """)
    
    # ProblemCluster -> RootCausePattern (HAS_ROOT_CAUSE)
    print("  Computing HAS_ROOT_CAUSE edges (ProblemCluster -> RootCausePattern)...")
    session.run("""
        MATCH (pc:ProblemCluster)<-[:BELONGS_TO]-(emr:EMRRecord)-[:CAUSED_BY]->(rc:RootCausePattern)
        WITH pc, rc, COUNT(emr) AS freq
        MERGE (pc)-[r:HAS_ROOT_CAUSE]->(rc)
        SET r.frequency = freq
    """)
    
    # RootCausePattern -> ActionPattern (RESOLVED_BY)
    print("  Computing RESOLVED_BY edges (RootCausePattern -> ActionPattern)...")
    session.run("""
        MATCH (rc:RootCausePattern)<-[:CAUSED_BY]-(emr:EMRRecord)-[:RESOLVED_BY]->(ap:ActionPattern)
        WITH rc, ap, COUNT(emr) AS freq
        MERGE (rc)-[r:RESOLVED_BY]->(ap)
        SET r.frequency = freq
    """)
    
    # ActionPattern -> Part (USES_PART)
    print("  Computing USES_PART edges (ActionPattern -> Part)...")
    session.run("""
        MATCH (ap:ActionPattern)<-[:RESOLVED_BY]-(emr:EMRRecord)-[:USED_PART]->(p:Part)
        WITH ap, p, COUNT(emr) AS freq
        MERGE (ap)-[r:USES_PART]->(p)
        SET r.frequency = freq
    """)
    
    # ActionPattern -> Component (TARGETS)
    print("  Computing TARGETS edges (ActionPattern -> Component)...")
    session.run("""
        MATCH (ap:ActionPattern)<-[:RESOLVED_BY]-(emr:EMRRecord)-[:AFFECTED]->(c:Component)
        WITH ap, c, COUNT(emr) AS freq
        MERGE (ap)-[r:TARGETS]->(c)
        SET r.frequency = freq
    """)
    
print("  Aggregate edges computed.")

5. Computing Aggregate Edges...
  Computing INDICATES edges (SymptomPattern -> ProblemCluster)...
  Computing HAS_ROOT_CAUSE edges (ProblemCluster -> RootCausePattern)...
  Computing RESOLVED_BY edges (RootCausePattern -> ActionPattern)...
  Computing USES_PART edges (ActionPattern -> Part)...
  Computing TARGETS edges (ActionPattern -> Component)...
  Aggregate edges computed.


In [7]:
print("6. Storing Embeddings on SymptomPattern Nodes...")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)

with driver.session() as session:
    result = session.run("MATCH (sp:SymptomPattern) RETURN sp.name AS name")
    symptom_names = [r["name"] for r in result]

print(f"  Embedding {len(symptom_names)} symptom patterns...")
sp_embeddings = model.encode(symptom_names, normalize_embeddings=True)

with driver.session() as session:
    for name, emb in zip(symptom_names, sp_embeddings):
        session.run(
            "MATCH (sp:SymptomPattern {name: $name}) SET sp.embedding = $emb",
            name=name, emb=emb.tolist()
        )

print(f"  Stored embeddings on {len(symptom_names)} SymptomPattern nodes.")

6. Storing Embeddings on SymptomPattern Nodes...


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8449.24it/s]


  Embedding 1096 symptom patterns...
  Stored embeddings on 1096 SymptomPattern nodes.


In [8]:
print("7. Graph Statistics...")

with driver.session() as session:
    # Node counts
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS type, COUNT(n) AS count
        ORDER BY count DESC
    """)
    print("\n  Node counts:")
    for r in result:
        print(f"    {r['type']}: {r['count']}")
    
    # Relationship counts
    result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS type, COUNT(r) AS count
        ORDER BY count DESC
    """)
    print("\n  Relationship counts:")
    for r in result:
        print(f"    {r['type']}: {r['count']}")
    
    # Sample causal chain
    result = session.run("""
        MATCH (sp:SymptomPattern)-[i:INDICATES]->(pc:ProblemCluster)-[hrc:HAS_ROOT_CAUSE]->(rc:RootCausePattern)-[cr:RESOLVED_BY]->(ap:ActionPattern)
        RETURN sp.name AS symptom, pc.label AS problem, rc.name AS cause, ap.name AS action, 
               i.frequency AS indicate_freq, hrc.frequency AS cause_freq, cr.frequency AS resolve_freq
        ORDER BY cr.frequency DESC
        LIMIT 10
    """)
    print("\n  Top 10 Causal Chains (Symptom -> Problem -> Root Cause -> Action):")
    for r in result:
        print(f"    {r['symptom']} -> {r['problem']} -> {r['cause']} -> {r['action']} (freq: {r['resolve_freq']})")

driver.close()
print("\nGraph ingestion complete! You can explore the graph at http://localhost:7474")

7. Graph Statistics...

  Node counts:
    EMRRecord: 20629
    Part: 4594
    SymptomPattern: 1096
    ProblemCluster: 1085
    ActionPattern: 941
    RootCausePattern: 625
    Component: 289
    MachineModel: 184

  Relationship counts:
    RESOLVED_BY: 29203
    ON_MACHINE: 20607
    CAUSED_BY: 19905
    EXHIBITED: 19565
    BELONGS_TO: 14519
    USED_PART: 14466
    USES_PART: 8565
    INDICATES: 5550
    HAS_ROOT_CAUSE: 4837
    AFFECTED: 4462
    TARGETS: 2040

  Top 10 Causal Chains (Symptom -> Problem -> Root Cause -> Action):
    FC Issue J23008 -> Track Frame Reinforce -> Kebocoran Seal Track Frame -> Reinforce Track Frame J23008 (freq: 141)
    Track Frame Reinforcement Required -> Track Frame Reinforce -> Kebocoran Seal Track Frame -> Reinforce Track Frame J23008 (freq: 141)
    Reinforce Track Frame Failure -> Track Frame Reinforce -> Kebocoran Seal Track Frame -> Reinforce Track Frame J23008 (freq: 141)
    Reinforce Track Frame Failure J23008 -> Track Frame Reinforcement